In [3]:
pip install pgmpy

In [4]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Create Bayesian Network
model = DiscreteBayesianNetwork([
    ('Burglary', 'Alarm'),
    ('Earthquake', 'Alarm'),
    ('Alarm', 'JohnCalls'),
    ('Alarm', 'MaryCalls')
])

# Define CPDs
cpd_burglary = TabularCPD(
    variable='Burglary',
    variable_card=2,
    values=[[0.001],
            [0.999]]
)

cpd_earthquake = TabularCPD(
    variable='Earthquake',
    variable_card=2,
    values=[[0.002],
            [0.998]]
)

cpd_alarm = TabularCPD(
    variable='Alarm',
    variable_card=2,
    values=[
        [0.95, 0.94, 0.29, 0.001],
        [0.05, 0.06, 0.71, 0.999]
    ],
    evidence=['Burglary', 'Earthquake'],
    evidence_card=[2, 2]
)

cpd_john = TabularCPD(
    variable='JohnCalls',
    variable_card=2,
    values=[
        [0.90, 0.05],
        [0.10, 0.95]
    ],
    evidence=['Alarm'],
    evidence_card=[2]
)

cpd_mary = TabularCPD(
    variable='MaryCalls',
    variable_card=2,
    values=[
        [0.70, 0.01],
        [0.30, 0.99]
    ],
    evidence=['Alarm'],
    evidence_card=[2]
)

# Add CPDs
model.add_cpds(
    cpd_burglary,
    cpd_earthquake,
    cpd_alarm,
    cpd_john,
    cpd_mary
)

# Check model
print("Model valid:", model.check_model())

# Inference
infer = VariableElimination(model)

# Calculate P(John=True, Mary=True, Alarm=True, Burglary=False, Earthquake=False)
result = infer.query(
    variables=['JohnCalls', 'MaryCalls', 'Alarm'],
    evidence={
        'Burglary': 1,
        'Earthquake': 1
    }
)

print("\nProbability of John=True, Mary=True, Alarm=True when no burglary and no earthquake:")
print(result)

# Direct joint probability calculation
probability = (
    0.999 *   # P(No Burglary)
    0.998 *   # P(No Earthquake)
    0.001 *   # P(Alarm | No Burglary, No Earthquake)
    0.90 *    # P(John Calls | Alarm)
    0.70      # P(Mary Calls | Alarm)
)

print("\nJoint Probability P(J,M,A,¬B,¬E) =", probability)

Model valid: True

Probability of John=True, Mary=True, Alarm=True when no burglary and no earthquake:
+--------------+--------------+----------+----------------------------------+
| JohnCalls    | MaryCalls    | Alarm    |   phi(JohnCalls,MaryCalls,Alarm) |
+==============+==============+==========+==================================+
| JohnCalls(0) | MaryCalls(0) | Alarm(0) |                           0.0006 |
+--------------+--------------+----------+----------------------------------+
| JohnCalls(0) | MaryCalls(0) | Alarm(1) |                           0.0005 |
+--------------+--------------+----------+----------------------------------+
| JohnCalls(0) | MaryCalls(1) | Alarm(0) |                           0.0003 |
+--------------+--------------+----------+----------------------------------+
| JohnCalls(0) | MaryCalls(1) | Alarm(1) |                           0.0495 |
+--------------+--------------+----------+----------------------------------+
| JohnCalls(1) | MaryCalls(0) | Alarm(0